# Notebook de Generación del Dataset de Entrenamiento

## 1. Selección del Dataset

Los datasets base seleccionados para la confección del dataset final para el entrenamiento de la IA de detección SMishing de Posdata son los siguientes:
- Kaggle SMS Spam Dataset (en inglés)
- Hugging Face softecapps's SMS Spam Dataset (en español, compuesto por un conjunto de entrenamiento y otro de test, aunque los usaremos como uno solo)

## 2. Traducción de los datasets

Puesto que contamos con mensajes tanto en inglés como en español, realizaremos un detectos que aprenda de ambos idiomas, de modo que el dataset en inglés será traducido también al español y, consiguientemente, el dataset en español será traducido al inglés.


### 2.1. Traducción al español del Dataset de Kaggle

In [ ]:
import pandas as pd
from transformers import MarianMTModel, MarianTokenizer
import torch
from tqdm import tqdm
from google.colab import drive
import os

# Constants
DRIVE_PATH = '/content/drive/My Drive/TFG_Posdata'
ENGLISH_FILE = os.path.join(DRIVE_PATH, 'datasets', 'en', 'spam_sms_english_kaggle.csv')
SPANISH_FILE = os.path.join(DRIVE_PATH, 'datasets', 'es', 'spam_sms_spanish_kaggle_raw.csv')
BATCH_SIZE = 100
NAMES = ['label', 'text']
MODEL_NAME = 'Helsinki-NLP/opus-mt-en-es'

# Auxiliary functions
def parse_broken_row(line):
  '''
  Parses a broken CSV row into label and text components.

  Parameters:
  ----------
  line : str
      A single line from the CSV file.
  
  Returns:
  -------
  tuple or None
      A tuple (label, text) if parsing is successful, otherwise None.
  '''
  line = line.strip()

  while line.endswith(';') or line.endswith(','):
    line = line[:-1]
  
  if line.startswith('"') and line.endswith('"'):
    line = line[1:-1]

  parts = line.split(',', 1)
  if len(parts) < 2:
    return None
  
  label = parts[0].strip()
  text = parts[1].strip()

  text = text.strip(' ,;')
  if text.startswith('""') and text.endswith('""'):
    text = text[2:-2].replace('""', '"')
  elif text.startswith('"') and text.endswith('"'):
    text = text[1:-1].replace('""', '"')  
  
  return label, text

def translate_batch(texts, tokenizer, model, device):
    '''
    Translates a batch of texts from English to Spanish.

    Parameters:
    ----------
    texts : list of str
        List of English texts to translate.
    tokenizer : MarianTokenizer
        The tokenizer for the translation model.
    model : MarianMTModel
        The translation model.
    device : str
        The device to run the model on ('cpu' or 'cuda').
    '''
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    with torch.no_grad():
        translated = model.generate(**inputs)
    return [tokenizer.decode(t, skip_special_tokens=True) for t in translated]


# Execution starts HERE

# Mount Google Drive
drive.mount('/content/drive')

data = [] # List to hold parsed data
try:
  with open(ENGLISH_FILE, 'r', encoding='latin-1') as file: # Open the dataset file
    for line in file: # Read each line
      parsed_row = parse_broken_row(line) # Parse the line
      if parsed_row: # If parsing was successful
        label, text = parsed_row # Unpack the tuple
        if label in ['ham', 'spam']: # Validate label
          data.append({'label': label, 'text': text}) # Append to data list
    dataset = pd.DataFrame(data) # Create DataFrame from data list
    print(f"Dataset loaded successfully. Total rows: {len(dataset)}")
except Exception as e:
    print(f"Error loading dataset: {e}")
    exit(1)

# Load translation model and tokenizer
print(f"Loading model {MODEL_NAME}...")
tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)
model = MarianMTModel.from_pretrained(MODEL_NAME)

# Move model to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Using device: {device}")

print("Starting translation...")

spanish_texts = [] # List to hold translated texts
total_texts = len(dataset) # Total number of texts to translate


for i in tqdm(range(0, total_texts, BATCH_SIZE)): # Process texts in batches
    batch_texts = dataset['text'].iloc[i:i + BATCH_SIZE].tolist() # Get batch of texts
    translated_texts = translate_batch(batch_texts, tokenizer, model, device) # Translate batch
    spanish_texts.extend(translated_texts) # Append translated texts to list

dataset['text_es'] = spanish_texts # Add translated texts to DataFrame

# Display some example translations
print("Example translations:")
print(dataset[['text', 'text_es']].head())

dataset_final = dataset[['label', 'text_es']].rename(columns={'text_es': 'text'}) # Prepare final dataset
dataset_final.to_csv(SPANISH_FILE, index=False) # Save to CSV

print(f"Translation completed. Translated dataset saved to {SPANISH_FILE}.")

In [ ]:
import html

# Constants
SPANISH_FILE_RAW = os.path.join(DRIVE_PATH, 'datasets', 'es', 'spam_sms_spanish_kaggle_raw.csv')
SPANISH_FILE_CLEAN = os.path.join(DRIVE_PATH, 'datasets', 'es', 'spam_sms_spanish_kaggle_clean.csv')

# Auxiliary function
def clean_text(text):
    '''
    Cleans text by unescaping HTML entities and fixing encoding issues.

    Parameters:
    ----------
    text : str
        The text to clean.

    Returns:
    -------
    str
        The cleaned text.
    '''
    if not isinstance(text, str):
        return str(text)
    
    text = html.unescape(text)
    
    try:
        text = text.encode('latin-1').decode('utf-8')
    except (UnicodeEncodeError, UnicodeDecodeError):
        pass
        
    return text

# Execution starts HERE

print(f"Loading {SPANISH_FILE_RAW}...")
df = pd.read_csv(SPANISH_FILE_RAW) # Load the raw Spanish dataset

print("Cleaning text...")
df['text'] = df['text'].apply(clean_text) # Clean the text column

df.to_csv(SPANISH_FILE_CLEAN, index=False, encoding='utf-8-sig') # Save cleaned dataset

print(f"Done! Cleaned dataset saved as: {SPANISH_FILE_CLEAN}")